# Validacao do conteudo do L2

### Validacao do spot 24x7

✔ Index é DatetimeIndex

✔ Timezone = UTC

✔ Ordenado crescente

✔ Sem timestamps duplicados

✔ Frequência exatamente 1 dia (24/7)

In [1]:
import os
import pandas as pd
from pathlib import Path

BASE_PATH = Path("/Users/brown/Documents/MLGeral/crypto_v2/crypto-market-state/data/02_intermediate/spot/daily")

ONE_DAY = pd.Timedelta(days=1)

def validate_asset(path: Path):
    df = pd.read_parquet(path)

    errors = []

    # Index checks
    if not isinstance(df.index, pd.DatetimeIndex):
        errors.append("Index is not DatetimeIndex")
    else:
        if df.index.tz is None or str(df.index.tz) != "UTC":
            errors.append("Index timezone is not UTC")

        if not df.index.is_monotonic_increasing:
            errors.append("Index not monotonic increasing")

        if df.index.duplicated().any():
            errors.append("Duplicate timestamps")

        if len(df) > 1:
            diffs = df.index.to_series().diff().dropna()
            if not (diffs == ONE_DAY).all():
                errors.append("Frequency is not exactly 1 day")

    # Required columns
    required_cols = ["open", "high", "low", "close", "volume", "trades", "close_time"]
    for col in required_cols:
        if col not in df.columns:
            errors.append(f"Missing column: {col}")

    # Dtypes
    for col in ["open", "high", "low", "close", "volume"]:
        if col in df.columns and df[col].dtype != "float64":
            errors.append(f"{col} not float64")

    if "trades" in df.columns and df["trades"].dtype != "int64":
        errors.append("trades not int64")

    # NaN check
    if df[required_cols].isna().any().any():
        errors.append("NaN in required columns")

    # OHLC integrity
    if not ((df["high"] >= df[["open", "close"]].max(axis=1)).all()):
        errors.append("High integrity violated")

    if not ((df["low"] <= df[["open", "close"]].min(axis=1)).all()):
        errors.append("Low integrity violated")

    if not (df["high"] >= df["low"]).all():
        errors.append("High < Low detected")

    if not (df[["open","high","low","close"]] > 0).all().all():
        errors.append("Non-positive price detected")

    # Volume integrity
    if (df["volume"] < 0).any():
        errors.append("Negative volume detected")

    if (df["trades"] < 0).any():
        errors.append("Negative trades detected")

    return errors


print("\n===== L2 SPOT VALIDATION =====\n")

for file in sorted(BASE_PATH.glob("*.parquet")):
    errors = validate_asset(file)
    asset = file.stem

    if errors:
        print(f"❌ {asset}")
        for e in errors:
            print("   -", e)
    else:
        print(f"✅ {asset} OK")

print("\nValidation completed.")


===== L2 SPOT VALIDATION =====

✅ ADAUSDT OK
✅ AVAXUSDT OK
✅ BNBUSDT OK
✅ BTCUSDT OK
✅ ETHUSDT OK
✅ LINKUSDT OK
✅ SOLUSDT OK
✅ XRPUSDT OK

Validation completed.


### validacao bd

In [4]:
import pandas as pd
from pathlib import Path

BASE_PATH = Path(
    "/Users/brown/Documents/MLGeral/crypto_v2/crypto-market-state/data/02_intermediate/spot/business_day"
)

REQUIRED_COLUMNS = {"open", "high", "low", "close", "volume"}
ONE_DAY = pd.Timedelta(days=1)
THREE_DAYS = pd.Timedelta(days=3)

print("\n===== L2 SPOT BUSINESS DAY VALIDATION =====")

for file in sorted(BASE_PATH.glob("*.parquet")):
    name = file.stem
    df = pd.read_parquet(file)

    try:
        # --- Index validation ---
        if not isinstance(df.index, pd.DatetimeIndex):
            raise ValueError("Index is not DatetimeIndex.")

        if df.index.tz is None or str(df.index.tz) != "UTC":
            raise ValueError("Index is not UTC.")

        if df.index.duplicated().any():
            raise ValueError("Duplicate timestamps found.")

        if not df.index.is_monotonic_increasing:
            raise ValueError("Index not sorted ascending.")

        # --- Required columns ---
        missing = REQUIRED_COLUMNS - set(df.columns)
        if missing:
            raise ValueError(f"Missing required columns: {missing}")

        # --- NaN check ---
        if df[list(REQUIRED_COLUMNS)].isna().any().any():
            raise ValueError("NaN detected in required columns.")

        # --- Object column check ---
        if df.select_dtypes(include=["object"]).shape[1] > 0:
            raise ValueError("Object dtype columns detected.")

        # --- Business day frequency ---
        if len(df) > 1:
            diffs = df.index.to_series().diff().dropna()
            valid = (diffs == ONE_DAY) | (diffs == THREE_DAYS)
            if not valid.all():
                raise ValueError("Invalid business-day gaps detected.")

        # --- OHLC integrity ---
        o, h, l, c = df["open"], df["high"], df["low"], df["close"]

        if ((h < o) | (h < c)).any():
            raise ValueError("High < max(open, close).")

        if ((l > o) | (l > c)).any():
            raise ValueError("Low > min(open, close).")

        if (h < l).any():
            raise ValueError("High < Low.")

        if (o <= 0).any() or (h <= 0).any() or (l <= 0).any() or (c <= 0).any():
            raise ValueError("Non-positive OHLC value detected.")

        if (df["volume"] < 0).any():
            raise ValueError("Negative volume detected.")

        print(f"✅ {name} OK")

    except Exception as e:
        print(f"❌ {name} FAILED → {e}")


===== L2 SPOT BUSINESS DAY VALIDATION =====
❌ gold FAILED → Invalid business-day gaps detected.
❌ nasdaq FAILED → Invalid business-day gaps detected.
❌ sp500 FAILED → Invalid business-day gaps detected.


In [6]:
# validacao 2 
import pandas as pd
from pathlib import Path

file_path = Path("/Users/brown/Documents/MLGeral/crypto_v2/crypto-market-state/data/02_intermediate/spot/business_day/gold.parquet")

print("📂 Arquivo existe:", file_path.exists())

df = pd.read_parquet(file_path)

print("\n===== HEAD =====")
display(df.head())

print("\n===== INFO =====")
print(df.info())

# -------------------------------------------------
# 1️⃣ Index validation
# -------------------------------------------------

print("\n===== INDEX CHECK =====")

assert isinstance(df.index, pd.DatetimeIndex), "Index não é DatetimeIndex"

print("✔ DatetimeIndex")

assert df.index.tz is not None and str(df.index.tz) == "UTC", "Index não está em UTC"

print("✔ UTC")

assert df.index.is_monotonic_increasing, "Index não é monotonicamente crescente"

print("✔ Monotonic increasing")

assert not df.index.duplicated().any(), "Existem timestamps duplicados"

print("✔ Sem duplicatas")

# -------------------------------------------------
# 2️⃣ Schema validation
# -------------------------------------------------

REQUIRED = {"open", "high", "low", "close", "volume"}

print("\n===== SCHEMA CHECK =====")

missing = REQUIRED - set(df.columns)

assert not missing, f"Colunas obrigatórias ausentes: {missing}"

print("✔ Colunas obrigatórias presentes")

# -------------------------------------------------
# 3️⃣ NaN check
# -------------------------------------------------

print("\n===== NaN CHECK =====")

assert not df[list(REQUIRED)].isna().any().any(), "Existem NaNs nas colunas obrigatórias"

print("✔ Sem NaNs")

# -------------------------------------------------
# 4️⃣ OHLC Integrity
# -------------------------------------------------

print("\n===== OHLC CHECK =====")

o, h, l, c = df["open"], df["high"], df["low"], df["close"]

assert not ((h < o) | (h < c)).any(), "High < max(open, close)"
assert not ((l > o) | (l > c)).any(), "Low > min(open, close)"
assert not (h < l).any(), "High < Low"
assert not ((o <= 0) | (h <= 0) | (l <= 0) | (c <= 0)).any(), "OHLC <= 0"

print("✔ Integridade OHLC OK")

# -------------------------------------------------
# 5️⃣ Volume integrity
# -------------------------------------------------

print("\n===== VOLUME CHECK =====")

assert not (df["volume"] < 0).any(), "Volume negativo"

print("✔ Volume >= 0")

# -------------------------------------------------
# 6️⃣ Object dtype check
# -------------------------------------------------

print("\n===== DTYPE CHECK =====")

obj_cols = df.select_dtypes(include=["object"]).columns.tolist()

assert len(obj_cols) == 0, f"Colunas object detectadas: {obj_cols}"

print("✔ Nenhuma coluna object")

print("\n🎯 L2 GOLD VALIDADO COM SUCESSO")

📂 Arquivo existe: True

===== HEAD =====


,close,high,low,open,volume
timestamp,,,,,
2020-10-01 00:00:00+00:00,1908.400024,1909.599976,1882.500000,1884.099976,730
2020-10-02 00:00:00+00:00,1900.199951,1913.000000,1893.900024,1893.900024,530
2020-10-05 00:00:00+00:00,1912.500000,1915.599976,1884.699951,1898.900024,1360
2020-10-06 00:00:00+00:00,1901.099976,1918.000000,1874.400024,1906.599976,968
2020-10-07 00:00:00+00:00,1883.599976,1889.800049,1873.099976,1874.099976,50



===== INFO =====
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1360 entries, 2020-10-01 00:00:00+00:00 to 2026-02-27 00:00:00+00:00
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   close   1360 non-null   float64
 1   high    1360 non-null   float64
 2   low     1360 non-null   float64
 3   open    1360 non-null   float64
 4   volume  1360 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 63.8 KB
None

===== INDEX CHECK =====
✔ DatetimeIndex
✔ UTC
✔ Monotonic increasing
✔ Sem duplicatas

===== SCHEMA CHECK =====
✔ Colunas obrigatórias presentes

===== NaN CHECK =====
✔ Sem NaNs

===== OHLC CHECK =====
✔ Integridade OHLC OK

===== VOLUME CHECK =====
✔ Volume >= 0

===== DTYPE CHECK =====
✔ Nenhuma coluna object

🎯 L2 GOLD VALIDADO COM SUCESSO


### validando L2 macro daily

In [7]:
import pandas as pd
from pathlib import Path

base_path = Path("/Users/brown/Documents/MLGeral/crypto_v2/crypto-market-state/data/02_intermediate/macro/daily")

files = sorted(base_path.glob("*.parquet"))

print(f"📂 Encontrados {len(files)} arquivos\n")

def validate_macro_daily(df: pd.DataFrame, name: str):
    if df is None or df.empty:
        raise ValueError("Partition is empty.")

    # Index
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("Index is not DatetimeIndex.")

    if df.index.tz is None or str(df.index.tz) != "UTC":
        raise ValueError("Index is not UTC.")

    if not df.index.is_monotonic_increasing:
        raise ValueError("Index not monotonic increasing.")

    if df.index.duplicated().any():
        raise ValueError("Duplicate timestamps detected.")

    # Structure
    if df.select_dtypes(include=["object"]).shape[1] > 0:
        raise ValueError("Object dtype columns detected.")

    if df.shape[1] == 0:
        raise ValueError("No value columns found.")

    # Numeric validation
    numeric_cols = df.columns
    for col in numeric_cols:
        if not pd.api.types.is_numeric_dtype(df[col]):
            raise ValueError(f"Column {col} is not numeric.")

    if df[numeric_cols].isna().all().any():
        raise ValueError("A column contains only NaN values.")

    return True


for file in files:
    try:
        df = pd.read_parquet(file)
        validate_macro_daily(df, file.name)
        print(f"✔ {file.name} OK")
    except Exception as e:
        print(f"❌ {file.name} FAILED → {e}")

print("\n🎯 Auditoria L2 Macro Daily concluída.")

📂 Encontrados 6 arquivos

✔ DGS10.parquet OK
✔ DGS2.parquet OK
✔ RRPONTSYD.parquet OK
✔ TEDRATE.parquet OK
✔ dxy.parquet OK
✔ vix.parquet OK

🎯 Auditoria L2 Macro Daily concluída.


### verificando macro monthly

In [8]:
import pandas as pd
from pathlib import Path

base_path = Path("/Users/brown/Documents/MLGeral/crypto_v2/crypto-market-state/data/02_intermediate/macro/monthly")

files = sorted(base_path.glob("*.parquet"))

print(f"📂 Encontrados {len(files)} arquivos\n")

def validate_macro_monthly(df: pd.DataFrame, name: str):

    if df is None or df.empty:
        raise ValueError("Partition is empty.")

    # -------- Index checks --------
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("Index is not DatetimeIndex.")

    if df.index.tz is None or str(df.index.tz) != "UTC":
        raise ValueError("Index is not UTC.")

    if not df.index.is_monotonic_increasing:
        raise ValueError("Index not monotonic increasing.")

    if df.index.duplicated().any():
        raise ValueError("Duplicate timestamps detected.")

    # -------- Structure checks --------
    if df.shape[1] == 0:
        raise ValueError("No value columns found.")

    if df.select_dtypes(include=["object"]).shape[1] > 0:
        raise ValueError("Object dtype columns detected.")

    # -------- Numeric checks --------
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            raise ValueError(f"Column {col} is not numeric.")

    # Column entirely NaN
    if df.isna().all().any():
        raise ValueError("A column contains only NaN values.")

    return True


for file in files:
    try:
        df = pd.read_parquet(file)
        validate_macro_monthly(df, file.name)
        print(f"✔ {file.name} OK")
    except Exception as e:
        print(f"❌ {file.name} FAILED → {e}")

print("\n🎯 Auditoria L2 Macro Monthly concluída.")

📂 Encontrados 5 arquivos

✔ CPIAUCSL.parquet OK
✔ FEDFUNDS.parquet OK
✔ INDPRO.parquet OK
✔ PAYEMS.parquet OK
✔ UNRATE.parquet OK

🎯 Auditoria L2 Macro Monthly concluída.


### validando macro wekkly

In [9]:
import os
import pandas as pd
from pathlib import Path

BASE_PATH = Path(
    "/Users/brown/Documents/MLGeral/crypto_v2/crypto-market-state/data/02_intermediate/macro/weekly"
)

print(f"\n📂 Pasta existe: {BASE_PATH.exists()}\n")

files = sorted(BASE_PATH.glob("*.parquet"))
print(f"📂 Encontrados {len(files)} arquivos\n")

def validate_macro_weekly(df: pd.DataFrame, name: str):
    errors = []

    # -------- Index checks --------
    if not isinstance(df.index, pd.DatetimeIndex):
        errors.append("Index não é DatetimeIndex")

    if df.index.tz is None or str(df.index.tz) != "UTC":
        errors.append("Index não está em UTC")

    if df.index.duplicated().any():
        errors.append("Index possui duplicatas")

    if not df.index.is_monotonic_increasing:
        errors.append("Index não é monotonic increasing")

    # -------- Estrutura --------
    if df.empty:
        errors.append("DataFrame vazio")

    if df.select_dtypes(include=["object"]).shape[1] > 0:
        errors.append("Possui colunas object")

    # -------- Colunas --------
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            errors.append(f"Coluna não numérica: {col}")

        if df[col].isna().all():
            errors.append(f"Coluna inteira NaN: {col}")

    return errors


# -------------------------
# Execução da auditoria
# -------------------------

for file in files:
    name = file.name
    df = pd.read_parquet(file)

    errs = validate_macro_weekly(df, name)

    if not errs:
        print(f"✔ {name} OK")
    else:
        print(f"\n❌ {name} FALHOU:")
        for e in errs:
            print("  -", e)

print("\n🎯 Auditoria L2 Macro Weekly concluída.")


📂 Pasta existe: True

📂 Encontrados 2 arquivos

✔ STLFSI4.parquet OK
✔ WALCL.parquet OK

🎯 Auditoria L2 Macro Weekly concluída.


### analise fred

In [3]:
import pandas as pd
import numpy as np

path = "/Users/brown/Documents/MLGeral/crypto_v2/crypto-market-state/data/01_raw/macro/daily/DGS2.parquet"

print("===== LOADING FILE =====")
df = pd.read_parquet(path)

print("\n===== BASIC INFO =====")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nDtypes:")
print(df.dtypes)

print("\n===== HEAD =====")
print(df.head())

print("\n===== TAIL =====")
print(df.tail())

print("\n===== DATE ANALYSIS =====")
if "date" in df.columns:
    date_col = "date"
elif "timestamp" in df.columns:
    date_col = "timestamp"
else:
    date_col = None

if date_col:
    print("Min date:", df[date_col].min())
    print("Max date:", df[date_col].max())
    print("Timezone:", df[date_col].dt.tz)
    print("Duplicated dates:", df[date_col].duplicated().sum())
else:
    print("No date/timestamp column found.")

print("\n===== NUMERIC SUMMARY =====")
print(df.describe(include="all"))

print("\n===== NULL CHECK =====")
print(df.isna().sum())

print("\n===== OBJECT COLUMNS =====")
print(df.select_dtypes(include=["object"]).columns.tolist())

===== LOADING FILE =====

===== BASIC INFO =====
Shape: (1406, 2)
Columns: ['date', 'value']

Dtypes:
date     datetime64[ns, UTC]
value                float64
dtype: object

===== HEAD =====
                       date  value
0 2020-10-01 00:00:00+00:00   0.14
1 2020-10-02 00:00:00+00:00   0.13
2 2020-10-05 00:00:00+00:00   0.14
3 2020-10-06 00:00:00+00:00   0.14
4 2020-10-07 00:00:00+00:00   0.16

===== TAIL =====
                          date  value
1401 2026-02-13 00:00:00+00:00   3.40
1402 2026-02-16 00:00:00+00:00    NaN
1403 2026-02-17 00:00:00+00:00   3.43
1404 2026-02-18 00:00:00+00:00   3.47
1405 2026-02-19 00:00:00+00:00   3.47

===== DATE ANALYSIS =====
Min date: 2020-10-01 00:00:00+00:00
Max date: 2026-02-19 00:00:00+00:00
Timezone: UTC
Duplicated dates: 0

===== NUMERIC SUMMARY =====
                                      date        value
count                                 1406  1344.000000
mean   2023-06-11 21:35:35.419630336+00:00     3.068028
min              2020-

### vix L1

In [4]:
import pandas as pd
import numpy as np

path = "/Users/brown/Documents/MLGeral/crypto_v2/crypto-market-state/data/01_raw/macro/daily/vix.parquet"

print("===== LOADING FILE =====")
df = pd.read_parquet(path)

print("\n===== BASIC INFO =====")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nDtypes:")
print(df.dtypes)

print("\n===== HEAD =====")
print(df.head())

print("\n===== TAIL =====")
print(df.tail())

print("\n===== DATE ANALYSIS =====")
if "date" in df.columns:
    date_col = "date"
elif "timestamp" in df.columns:
    date_col = "timestamp"
else:
    date_col = None

if date_col:
    print("Min date:", df[date_col].min())
    print("Max date:", df[date_col].max())
    print("Timezone:", df[date_col].dt.tz)
    print("Duplicated dates:", df[date_col].duplicated().sum())
else:
    print("No date/timestamp column found.")

print("\n===== NUMERIC SUMMARY =====")
print(df.describe(include="all"))

print("\n===== NULL CHECK =====")
print(df.isna().sum())

print("\n===== OBJECT COLUMNS =====")
print(df.select_dtypes(include=["object"]).columns.tolist())

===== LOADING FILE =====

===== BASIC INFO =====
Shape: (1969, 6)
Columns: ['date', 'close', 'high', 'low', 'open', 'volume']

Dtypes:
date      datetime64[ns, UTC]
close                 float64
high                  float64
low                   float64
open                  float64
volume                  int64
dtype: object

===== HEAD =====
                       date      close       high    low       open  volume
0 2020-10-01 00:00:00+00:00  26.700001  27.110001  25.33  25.780001       0
1 2020-10-02 00:00:00+00:00  27.629999  29.900000  26.93  28.870001       0
2 2020-10-03 00:00:00+00:00  27.629999  29.900000  26.93  28.870001       0
3 2020-10-04 00:00:00+00:00  27.629999  29.900000  26.93  28.870001       0
4 2020-10-05 00:00:00+00:00  27.959999  29.690001  27.27  29.520000       0

===== TAIL =====
                          date      close       high    low       open  volume
1964 2026-02-16 00:00:00+00:00  20.600000  22.400000  18.92  21.480000       0
1965 2026-02-17 00:00

### identificando qual ativo esta com dados nos finais de semana

In [1]:
import os
import pandas as pd
from pathlib import Path

BASE_PATH = Path("/Users/brown/Documents/MLGeral/crypto_v2/crypto-market-state/data/01_raw/spot/business_day")

ONE_DAY = pd.Timedelta(days=1)
THREE_DAYS = pd.Timedelta(days=3)

results = []

for file in sorted(BASE_PATH.glob("*.parquet")):
    asset_name = file.stem
    
    try:
        df = pd.read_parquet(file)
    except Exception as e:
        print(f"Erro ao carregar {asset_name}: {e}")
        continue
    
    # Detectar coluna de data
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], utc=True)
        df = df.sort_values("date").set_index("date")
    elif "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
        df = df.sort_values("timestamp").set_index("timestamp")
    else:
        print(f"{asset_name} ❌ não possui coluna date/timestamp")
        continue
    
    diffs = df.index.to_series().diff().dropna()
    invalid = diffs[(diffs != ONE_DAY) & (diffs != THREE_DAYS)]
    
    if not invalid.empty:
        results.append({
            "asset": asset_name,
            "num_breaks": len(invalid),
            "first_break": invalid.index[0],
            "delta": invalid.iloc[0]
        })
        
        print(f"\n❌ {asset_name} QUEBRA CONTRATO")
        print(invalid.head())
    else:
        print(f"✅ {asset_name} OK")

print("\n==============================")
print("RESUMO")
print("==============================")

if results:
    summary = pd.DataFrame(results)
    display(summary)
else:
    print("Todos os ativos respeitam 1d / 3d.")


❌ gold QUEBRA CONTRATO
date
2020-11-27 00:00:00+00:00   2 days
2020-12-28 00:00:00+00:00   4 days
2021-01-04 00:00:00+00:00   4 days
2021-01-19 00:00:00+00:00   4 days
2021-02-16 00:00:00+00:00   4 days
Name: date, dtype: timedelta64[ns]

❌ nasdaq QUEBRA CONTRATO
date
2020-11-27 00:00:00+00:00   2 days
2020-12-28 00:00:00+00:00   4 days
2021-01-04 00:00:00+00:00   4 days
2021-01-19 00:00:00+00:00   4 days
2021-02-16 00:00:00+00:00   4 days
Name: date, dtype: timedelta64[ns]

❌ sp500 QUEBRA CONTRATO
date
2020-11-27 00:00:00+00:00   2 days
2020-12-28 00:00:00+00:00   4 days
2021-01-04 00:00:00+00:00   4 days
2021-01-19 00:00:00+00:00   4 days
2021-02-16 00:00:00+00:00   4 days
Name: date, dtype: timedelta64[ns]

RESUMO


,asset,num_breaks,first_break,delta
0,gold,52,2020-11-27 00:00:00+00:00,2 days
1,nasdaq,54,2020-11-27 00:00:00+00:00,2 days
2,sp500,54,2020-11-27 00:00:00+00:00,2 days


### verificando futuros funding e perpetual

In [10]:
import pandas as pd
from pathlib import Path

# =========================================================
# Paths
# =========================================================

funding_path = Path(
    "/Users/brown/Documents/MLGeral/crypto_v2/crypto-market-state/data/01_raw/futures/crypto/funding/BTCUSDT.parquet"
)

perpetual_path = Path(
    "/Users/brown/Documents/MLGeral/crypto_v2/crypto-market-state/data/01_raw/futures/crypto/perpetual/1h/BTCUSDT.parquet"
)

# =========================================================
# Generic validator
# =========================================================

def validate_basic_l1(df: pd.DataFrame, name: str):
    print(f"\n{'='*70}")
    print(f"🔎 VALIDANDO {name}")
    print(f"{'='*70}\n")

    print("HEAD:\n", df.head(), "\n")
    print("INFO:")
    print(df.info(), "\n")

    # -------- Index check --------
    if isinstance(df.index, pd.DatetimeIndex):
        print("✔ Index é DatetimeIndex")
        print("✔ TZ:", df.index.tz)
        print("✔ Monotonic:", df.index.is_monotonic_increasing)
        print("✔ Duplicatas:", df.index.duplicated().any())
    else:
        print("ℹ Index não é DatetimeIndex (ok se timestamp for coluna L1)")

    # -------- Object columns --------
    obj_cols = df.select_dtypes(include=["object"]).columns
    if len(obj_cols) > 0:
        print("⚠ Colunas object:", list(obj_cols))
    else:
        print("✔ Sem colunas object")

    print("\nResumo estatístico:")
    print(df.describe(include="all"))


# =========================================================
# Funding L1 Contract Check
# =========================================================

def validate_funding_l1(df: pd.DataFrame):
    print("\n🧠 Verificando contrato L1 Funding")

    expected_columns = {"open_time", "fundingRate", "markPrice", "symbol", "extracted_at"}
    missing = expected_columns - set(df.columns)

    if missing:
        print("❌ Colunas esperadas ausentes:", missing)
    else:
        print("✔ Colunas principais presentes")

    if "open_time" in df.columns:
        ts = pd.to_datetime(df["open_time"], utc=True)
        print("✔ open_time convertido para datetime UTC")

        diffs = ts.diff().dropna()
        print("Intervalo médio entre registros:", diffs.mode().iloc[0] if not diffs.empty else "N/A")

    if "fundingRate" in df.columns:
        print("Funding rate stats:")
        print(df["fundingRate"].describe())


# =========================================================
# Perpetual 1h L1 Contract Check
# =========================================================

def validate_perpetual_l1(df: pd.DataFrame):
    print("\n🧠 Verificando contrato L1 Perpetual 1h")

    expected_ohlcv = {"open_time", "open", "high", "low", "close", "volume"}
    missing = expected_ohlcv - set(df.columns)

    if missing:
        print("❌ Colunas OHLCV ausentes:", missing)
    else:
        print("✔ Colunas OHLCV presentes")

    if "open_time" in df.columns:
        ts = pd.to_datetime(df["open_time"], utc=True)
        diffs = ts.diff().dropna()

        if not diffs.empty:
            print("Intervalo dominante:", diffs.mode().iloc[0])
        else:
            print("Sem dados suficientes para checar intervalo")

    # OHLC sanity
    if all(col in df.columns for col in ["open", "high", "low", "close"]):
        o, h, l, c = df["open"], df["high"], df["low"], df["close"]

        print("High < Low ocorrências:", (h < l).sum())
        print("High < max(open,close):", ((h < o) | (h < c)).sum())
        print("Low > min(open,close):", ((l > o) | (l > c)).sum())


# =========================================================
# Execute
# =========================================================

if funding_path.exists():
    funding_df = pd.read_parquet(funding_path)
    validate_basic_l1(funding_df, "FUNDING BTCUSDT")
    validate_funding_l1(funding_df)
else:
    print("❌ Funding file não encontrado")

if perpetual_path.exists():
    perp_df = pd.read_parquet(perpetual_path)
    validate_basic_l1(perp_df, "PERPETUAL BTCUSDT 1H")
    validate_perpetual_l1(perp_df)
else:
    print("❌ Perpetual file não encontrado")

print("\n🎯 Auditoria L1 Futures concluída.")


🔎 VALIDANDO FUNDING BTCUSDT

HEAD:
     symbol                        open_time  fundingRate  markPrice  \
0  BTCUSDT        2020-10-01 00:00:00+00:00     0.000074        NaN   
1  BTCUSDT        2020-10-01 08:00:00+00:00     0.000100        NaN   
2  BTCUSDT        2020-10-01 16:00:00+00:00     0.000100        NaN   
3  BTCUSDT 2020-10-02 00:00:00.011000+00:00    -0.000003        NaN   
4  BTCUSDT        2020-10-02 08:00:00+00:00    -0.000017        NaN   

                      extracted_at  
0 2026-02-27 22:47:59.160668+00:00  
1 2026-02-27 22:47:59.160668+00:00  
2 2026-02-27 22:47:59.160668+00:00  
3 2026-02-27 22:47:59.160668+00:00  
4 2026-02-27 22:47:59.160668+00:00   

INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5928 entries, 0 to 5927
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   symbol        5928 non-null   object             
 1   open_time     5928 non-null   

### Comparando L1 e L2 para BTC

In [11]:
import pandas as pd
from pathlib import Path

# ==============================
# PATHS
# ==============================

l1_path = Path("/Users/brown/Documents/MLGeral/crypto_v2/crypto-market-state/data/01_raw/spot/crypto/daily_24x7/BTCUSDT.parquet")
l2_path = Path("/Users/brown/Documents/MLGeral/crypto_v2/crypto-market-state/data/02_intermediate/spot/daily/BTCUSDT.parquet")

print("=" * 70)
print("📂 EXISTENCE CHECK")
print("=" * 70)
print("L1 exists:", l1_path.exists())
print("L2 exists:", l2_path.exists())

# ==============================
# LOAD
# ==============================

df_l1 = pd.read_parquet(l1_path)
df_l2 = pd.read_parquet(l2_path)

print("\n" + "=" * 70)
print("🔎 L1 HEAD")
print("=" * 70)
print(df_l1.head())

print("\n" + "=" * 70)
print("🔎 L2 HEAD")
print("=" * 70)
print(df_l2.head())

print("\n" + "=" * 70)
print("📊 L1 INFO")
print("=" * 70)
print(df_l1.info())

print("\n" + "=" * 70)
print("📊 L2 INFO")
print("=" * 70)
print(df_l2.info())

# ==============================
# COLUMN COMPARISON
# ==============================

print("\n" + "=" * 70)
print("🧱 COLUMN COMPARISON")
print("=" * 70)

print("L1 columns:", list(df_l1.columns))
print("L2 columns:", list(df_l2.columns))

# ==============================
# INDEX / TIMESTAMP CHECK
# ==============================

print("\n" + "=" * 70)
print("⏱ TEMPORAL CHECK")
print("=" * 70)

# L1 timestamp column name detection
if "open_time" in df_l1.columns:
    ts_l1 = pd.to_datetime(df_l1["open_time"], utc=True)
elif "timestamp" in df_l1.columns:
    ts_l1 = pd.to_datetime(df_l1["timestamp"], utc=True)
elif "date" in df_l1.columns:
    ts_l1 = pd.to_datetime(df_l1["date"], utc=True)
else:
    raise ValueError("L1: No temporal column found.")

ts_l2 = df_l2.index if isinstance(df_l2.index, pd.DatetimeIndex) else pd.to_datetime(df_l2["timestamp"], utc=True)

print("L1 rows:", len(ts_l1))
print("L2 rows:", len(ts_l2))

print("L1 min/max:", ts_l1.min(), "→", ts_l1.max())
print("L2 min/max:", ts_l2.min(), "→", ts_l2.max())

print("L2 monotonic:", ts_l2.is_monotonic_increasing)
print("L2 duplicates:", ts_l2.duplicated().any())

# ==============================
# VALUE CONSISTENCY CHECK
# ==============================

print("\n" + "=" * 70)
print("🔬 VALUE CONSISTENCY CHECK (OHLCV)")
print("=" * 70)

common_cols = ["open", "high", "low", "close", "volume"]

if "open_time" in df_l1.columns:
    df_l1 = df_l1.rename(columns={"open_time": "timestamp"})

df_l1["timestamp"] = pd.to_datetime(df_l1["timestamp"], utc=True)
df_l1 = df_l1.set_index("timestamp").sort_index()

df_l2_sorted = df_l2.sort_index()

aligned = df_l1[common_cols].join(
    df_l2_sorted[common_cols],
    lsuffix="_l1",
    rsuffix="_l2",
    how="inner"
)

differences = {}

for col in common_cols:
    diff = (aligned[f"{col}_l1"] - aligned[f"{col}_l2"]).abs().sum()
    differences[col] = diff

print("Total absolute difference per column:")
for k, v in differences.items():
    print(f"{k}: {v}")

if all(v == 0 for v in differences.values()):
    print("\n🎯 L2 preservou exatamente os valores do L1.")
else:
    print("\n⚠️ Existem diferenças entre L1 e L2.")

📂 EXISTENCE CHECK
L1 exists: True
L2 exists: True

🔎 L1 HEAD
                  open_time      open      high       low     close  \
0 2020-10-01 00:00:00+00:00  10776.59  10920.00  10437.00  10619.13   
1 2020-10-02 00:00:00+00:00  10619.13  10664.64  10374.00  10570.40   
2 2020-10-03 00:00:00+00:00  10570.40  10603.56  10496.46  10542.06   
3 2020-10-04 00:00:00+00:00  10542.07  10696.87  10517.87  10666.63   
4 2020-10-05 00:00:00+00:00  10666.62  10798.00  10615.64  10792.21   

         volume                       close_time  quote_volume  trades  \
0  60866.332893 2020-10-01 23:59:59.999000+00:00  6.521690e+08  794855   
1  50130.393705 2020-10-02 23:59:59.999000+00:00  5.268644e+08  777193   
2  22298.221341 2020-10-03 23:59:59.999000+00:00  2.351609e+08  381329   
3  23212.001595 2020-10-04 23:59:59.999000+00:00  2.462521e+08  377553   
4  34025.761653 2020-10-05 23:59:59.999000+00:00  3.641585e+08  483340   

   taker_buy_base_volume  taker_buy_quote_volume  
0           2844